In [ ]:
import random
import time
import uuid
from pathlib import Path

import httpx
import msal


def find_lab1_sidecar_env() -> Path:
    for env_path in (
        Path.cwd() / "Lab1" / "sidecar" / ".env",
        Path.cwd().parent.parent / "Lab1" / "sidecar" / ".env",
        Path.cwd().parent / "Lab1" / "sidecar" / ".env",
    ):
        if env_path.is_file():
            return env_path.resolve()
    raise FileNotFoundError(
        "Could not find Lab1/sidecar/.env — complete Lab1 setup first "
        "(notebook cwd: repo root or Lab2/scripts)."
    )


def parse_dotenv(path: Path) -> dict[str, str]:
    values: dict[str, str] = {}
    for line in path.read_text(encoding="utf-8").splitlines():
        line = line.strip()
        if not line or line.startswith("#"):
            continue
        if "=" not in line:
            continue
        key, _, value = line.partition("=")
        values[key.strip()] = value.strip().strip('"').strip("'")
    return values


def require_lab1_env_value(env: dict[str, str], key: str) -> str:
    value = env.get(key, "").strip()
    if not value or value.startswith("your-"):
        raise ValueError(f"Set {key} in Lab1/sidecar/.env (from Lab1 entra-agent-id-setup.ipynb).")
    return value


_lab1_env_path = find_lab1_sidecar_env()
_lab1_env = parse_dotenv(_lab1_env_path)
print(f"Loaded Lab1 sidecar env: {_lab1_env_path}")

TENANT_ID = require_lab1_env_value(_lab1_env, "TENANT_ID")
BLUEPRINT_APP_ID = require_lab1_env_value(_lab1_env, "BLUEPRINT_APP_ID")
AGENT_CLIENT_ID = require_lab1_env_value(_lab1_env, "AGENT_CLIENT_ID")
WEATHER_API_APP_ID = require_lab1_env_value(_lab1_env, "WEATHER_API_APP_ID")

WEATHER_DELEGATED_SCOPE_VALUE = "weather.read"

RUN_SUFFIX = random.randint(100000, 999999)
APP_DISPLAY_NAME = f"Agent Demo Client SPA {RUN_SUFFIX}"
REDIRECT_URI = "http://localhost:3003"
CLIENT_SPA_APP_ID = None

GRAPH_ROOT = "https://graph.microsoft.com/v1.0"
GRAPH_BETA_ROOT = "https://graph.microsoft.com/beta"
GRAPH_POWERSHELL_APP_CLIENT_ID = "14d82eec-204b-4c2f-b7e8-296a70dab67e"

DELEGATED_SCOPES = [
    "https://graph.microsoft.com/User.Read",
    "https://graph.microsoft.com/Application.ReadWrite.All",
    "https://graph.microsoft.com/AgentIdentityBlueprint.ReadWrite.All",
    "https://graph.microsoft.com/DelegatedPermissionGrant.ReadWrite.All",
]

if "00000000" in TENANT_ID:
    raise ValueError("Set TENANT_ID in the configuration cell before continuing.")
print(f"Run suffix: {RUN_SUFFIX}")
print(f"SPA display name: {APP_DISPLAY_NAME}")
print(f"Redirect URI: {REDIRECT_URI}")
if BLUEPRINT_APP_ID:
    print(f"Blueprint app id (Part 2): {BLUEPRINT_APP_ID}")
else:
    print("BLUEPRINT_APP_ID not set yet — required before Part 2")
if AGENT_CLIENT_ID:
    print(f"Agent app id: {AGENT_CLIENT_ID}")
else:
    print("AGENT_CLIENT_ID not set yet — required before Part 2")
if WEATHER_API_APP_ID:
    print(f"Weather API app id (Part 3): {WEATHER_API_APP_ID}")
else:
    print("WEATHER_API_APP_ID not set yet — required before Part 3")

In [ ]:
def acquire_delegated_token_device_code(
    tenant_id: str, scopes: list[str], client_id: str
) -> str:
    """Device code flow for Graph app registration (same model as Connect-MgGraph)."""
    authority = f"https://login.microsoftonline.com/{tenant_id}"
    app = msal.PublicClientApplication(client_id, authority=authority)
    flow = app.initiate_device_flow(scopes=scopes)
    if "user_code" not in flow:
        raise RuntimeError(f"Device flow failed: {flow}")
    print(flow["message"])
    result = app.acquire_token_by_device_flow(flow)
    if "access_token" not in result:
        raise RuntimeError(f"Sign-in failed: {result.get('error_description', result)}")
    return result["access_token"]


def graph_request(
    method: str,
    url: str,
    token: str,
    json_body: dict | None = None,
) -> httpx.Response:
    headers = {
        "Authorization": f"Bearer {token}",
        "Content-Type": "application/json",
        "OData-Version": "4.0",
    }
    with httpx.Client(timeout=60.0) as client:
        response = client.request(method, url, headers=headers, json=json_body)
    if response.status_code >= 400:
        raise RuntimeError(f"{method} {url} -> {response.status_code}: {response.text}")
    return response


## Admin sign-in (device code)

Run the cell below and complete sign-in in your browser (URL + user code). If your tenant blocks the Graph PowerShell public client, register your own app and replace `GRAPH_POWERSHELL_APP_CLIENT_ID` in the config cell.


In [ ]:
admin_token = acquire_delegated_token_device_code(
    TENANT_ID, DELEGATED_SCOPES, GRAPH_POWERSHELL_APP_CLIENT_ID
)
me = graph_request("GET", f"{GRAPH_ROOT}/me", admin_token).json()
print(f"Signed in as: {me.get('userPrincipalName', me.get('displayName'))}")


## Step 1 — Find or create SPA app registration

Matches PowerShell: find by `displayName`, or create with single-tenant audience and SPA redirect URIs.


In [ ]:
client_spa_app_id = CLIENT_SPA_APP_ID

if client_spa_app_id:
    lookup_url = (
        f"{GRAPH_ROOT}/applications?$filter=appId eq '{client_spa_app_id}'"
        "&$select=id,appId,displayName"
    )
    lookup = graph_request("GET", lookup_url, admin_token).json()
    if not lookup.get("value"):
        raise RuntimeError(f"No application for CLIENT_SPA_APP_ID={client_spa_app_id}")
    print(f"Using existing app: {lookup['value'][0]['displayName']} ({client_spa_app_id})")
else:
    safe_name = APP_DISPLAY_NAME.replace("'", "''")
    filter_url = (
        f"{GRAPH_ROOT}/applications?$filter=displayName eq '{safe_name}'"
        "&$select=id,appId,displayName"
    )
    existing = graph_request("GET", filter_url, admin_token).json()
    if existing.get("value"):
        client_spa_app_id = existing["value"][0]["appId"]
        print(f"Already exists: {client_spa_app_id}")
    else:
        create_body = {
            "displayName": APP_DISPLAY_NAME,
            "signInAudience": "AzureADMyOrg",
            "spa": {"redirectUris": [REDIRECT_URI]},
        }
        created = graph_request(
            "POST", f"{GRAPH_ROOT}/applications", admin_token, create_body
        ).json()
        client_spa_app_id = created["appId"]
        print(f"Created: {client_spa_app_id}")
        print("Waiting 5s for propagation...")
        time.sleep(5)

print(f"Client SPA App ID: {client_spa_app_id}")


## Step 2 — Set SPA redirect URI


In [ ]:
spa_body = {"spa": {"redirectUris": [REDIRECT_URI]}}
graph_request(
    "PATCH",
    f"{GRAPH_ROOT}/applications(appId='{client_spa_app_id}')",
    admin_token,
    spa_body,
)
print(f"Redirect URI set: {REDIRECT_URI}")


## Step 3 — Ensure service principal


In [ ]:
sp_url = (
    f"{GRAPH_ROOT}/servicePrincipals?$filter=appId eq '{client_spa_app_id}'&$select=id"
)
sp_result = graph_request("GET", sp_url, admin_token).json()
if sp_result.get("value"):
    print("Service principal already exists")
else:
    graph_request(
        "POST",
        f"{GRAPH_ROOT}/servicePrincipals",
        admin_token,
        {"appId": client_spa_app_id},
    )
    print("Created service principal")


In [9]:
print("Part 1 complete")
print(f"  CLIENT_SPA_APP_ID={client_spa_app_id}")
print(f"  Redirect URI: {REDIRECT_URI}")

Part 1 complete
  CLIENT_SPA_APP_ID=acec2210-912f-4cc0-a922-8d7b67a38793
  Redirect URI: http://localhost:3003


In [8]:
def find_lab2_sidecar_env() -> Path:
    for sidecar_dir in (
        Path.cwd() / "Lab2" / "sidecar",
        Path.cwd().parent / "sidecar",
        Path.cwd().parent.parent / "Lab2" / "sidecar",
    ):
        if (sidecar_dir / "docker-compose.yml").is_file():
            return (sidecar_dir / ".env").resolve()
    raise FileNotFoundError(
        "Could not find Lab2/sidecar/.env — copy Lab2/sidecar/.env.example to .env first."
    )


def update_dotenv(path: Path, updates: dict[str, str]) -> None:
    lines: list[str] = []
    if path.is_file():
        lines = path.read_text(encoding="utf-8").splitlines()

    remaining = dict(updates)
    new_lines: list[str] = []
    for line in lines:
        matched = False
        stripped = line.strip()
        for key, value in list(remaining.items()):
            if stripped.startswith(f"{key}=") or stripped == key:
                new_lines.append(f"{key}={value}")
                del remaining[key]
                matched = True
                break
        if not matched:
            new_lines.append(line)

    for key, value in remaining.items():
        if new_lines and new_lines[-1] != "":
            new_lines.append("")
        new_lines.append(f"{key}={value}")

    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text("\n".join(new_lines) + "\n", encoding="utf-8")


lab2_env_path = find_lab2_sidecar_env()
update_dotenv(lab2_env_path, {"CLIENT_SPA_APP_ID": client_spa_app_id})
print(f"Updated {lab2_env_path}")
print(f"  CLIENT_SPA_APP_ID={client_spa_app_id}")

Updated C:\develop\open-source\entra-agent-id-sidecar-lab\Lab2\sidecar\.env
  CLIENT_SPA_APP_ID=acec2210-912f-4cc0-a922-8d7b67a38793


Part 2 — Blueprint OBO setup

In [10]:
if not BLUEPRINT_APP_ID:
    raise ValueError("Set BLUEPRINT_APP_ID in the configuration cell before Part 2.")
if not AGENT_CLIENT_ID:
    raise ValueError("Set AGENT_CLIENT_ID in the configuration cell (Lab1 sidecar .env).")

print(f"Blueprint:  {BLUEPRINT_APP_ID}")
print(f"Agent:      {AGENT_CLIENT_ID}  (notebook does not grant Graph consent; PS step 5 only)")
print(f"Client SPA: {client_spa_app_id}")

agent_sp = graph_request(
    "GET",
    f"{GRAPH_ROOT}/servicePrincipals?$filter=appId eq '{AGENT_CLIENT_ID}'&$select=id,displayName",
    admin_token,
).json()
if not agent_sp.get("value"):
    raise RuntimeError(f"No service principal for AGENT_CLIENT_ID={AGENT_CLIENT_ID}")
print(f"Agent SP ok: {agent_sp['value'][0].get('displayName')}")

Blueprint:  c5eed854-4755-4edc-a0d2-f4055bbe6765
Agent:      1e3bcf56-d3ea-46de-b7d2-08036fe140cf  (notebook does not grant Graph consent; PS step 5 only)
Client SPA: acec2210-912f-4cc0-a922-8d7b67a38793
Agent SP ok: Lab1 Weather Agent 843509


In [11]:
lookup_url = (
    f"{GRAPH_BETA_ROOT}/applications?$filter=appId eq '{BLUEPRINT_APP_ID}'"
    "&$select=id,displayName,identifierUris,api"
)
result = graph_request("GET", lookup_url, admin_token).json()
if not result.get("value"):
    raise RuntimeError(f"No blueprint application for BLUEPRINT_APP_ID={BLUEPRINT_APP_ID}")

blueprint_app = result["value"][0]
blueprint_object_id = blueprint_app["id"]
existing_scopes = (blueprint_app.get("api") or {}).get("oauth2PermissionScopes") or []

print(f"Found: {blueprint_app.get('displayName')}")
print(f"Object ID: {blueprint_object_id}")
print(f"Current URIs: {', '.join(blueprint_app.get('identifierUris') or []) or '(none)'}")
scope_values = [s.get("value") for s in existing_scopes]
print(f"Current scopes: {', '.join(scope_values) or '(none)'}")

Found: Lab1 Agent Blueprint 843509
Object ID: c5eed854-4755-4edc-a0d2-f4055bbe6765
Current URIs: (none)
Current scopes: (none)


In [12]:
app_id_uri = f"api://{BLUEPRINT_APP_ID}"
print(f"Application ID URI → {app_id_uri}")

identifier_uris = blueprint_app.get("identifierUris") or []
if app_id_uri in identifier_uris:
    print("Already set")
else:
    graph_request(
        "PATCH",
        f"{GRAPH_BETA_ROOT}/applications/{blueprint_object_id}",
        admin_token,
        {"identifierUris": [app_id_uri]},
    )
    print(f"Set App ID URI to {app_id_uri}")

Application ID URI → api://c5eed854-4755-4edc-a0d2-f4055bbe6765
Set App ID URI to api://c5eed854-4755-4edc-a0d2-f4055bbe6765


In [13]:
has_access_as_user = next(
    (s for s in existing_scopes if s.get("value") == "access_as_user"),
    None,
)

if has_access_as_user:
    access_as_user_scope_id = has_access_as_user["id"]
    print(f"Scope already exists (ID: {access_as_user_scope_id})")
else:
    access_as_user_scope_id = str(uuid.uuid4())
    scope_body = {
        "api": {
            "oauth2PermissionScopes": [
                {
                    "id": access_as_user_scope_id,
                    "adminConsentDescription": "Allow the application to access the agent on behalf of the signed-in user",
                    "adminConsentDisplayName": "Access agent as user",
                    "isEnabled": True,
                    "type": "User",
                    "userConsentDescription": "Allow the application to access the agent on your behalf",
                    "userConsentDisplayName": "Access agent as user",
                    "value": "access_as_user",
                }
            ]
        }
    }
    graph_request(
        "PATCH",
        f"{GRAPH_BETA_ROOT}/applications/{blueprint_object_id}",
        admin_token,
        scope_body,
    )
    print(f"Added access_as_user scope (ID: {access_as_user_scope_id})")
    print(f"Full scope: {app_id_uri}/access_as_user")

Added access_as_user scope (ID: 4533103c-b6d5-4689-9b8f-858d37b7da5f)
Full scope: api://c5eed854-4755-4edc-a0d2-f4055bbe6765/access_as_user


In [14]:
bp_updated = graph_request(
    "GET",
    f"{GRAPH_BETA_ROOT}/applications?$filter=appId eq '{BLUEPRINT_APP_ID}'&$select=id,api",
    admin_token,
).json()
scopes = (bp_updated["value"][0].get("api") or {}).get("oauth2PermissionScopes") or []
scope_id = next((s["id"] for s in scopes if s.get("value") == "access_as_user"), None)

spa_lookup = graph_request(
    "GET",
    f"{GRAPH_ROOT}/applications?$filter=appId eq '{client_spa_app_id}'&$select=id",
    admin_token,
).json()
client_spa_object_id = spa_lookup["value"][0]["id"] if spa_lookup.get("value") else None

if scope_id and client_spa_object_id:
    perm_body = {
        "requiredResourceAccess": [
            {
                "resourceAppId": BLUEPRINT_APP_ID,
                "resourceAccess": [{"id": scope_id, "type": "Scope"}],
            }
        ]
    }
    graph_request(
        "PATCH",
        f"{GRAPH_ROOT}/applications/{client_spa_object_id}",
        admin_token,
        perm_body,
    )
    print("Added API permission on Client SPA → Blueprint access_as_user")
else:
    print(f"Missing scope_id={scope_id} or client_spa_object_id={client_spa_object_id}")

Added API permission on Client SPA → Blueprint access_as_user


In [15]:
blueprint_sp = graph_request(
    "GET",
    f"{GRAPH_ROOT}/servicePrincipals?$filter=appId eq '{BLUEPRINT_APP_ID}'&$select=id",
    admin_token,
).json()
blueprint_sp_id = blueprint_sp["value"][0]["id"] if blueprint_sp.get("value") else None

client_spa_sp = graph_request(
    "GET",
    f"{GRAPH_ROOT}/servicePrincipals?$filter=appId eq '{client_spa_app_id}'&$select=id",
    admin_token,
).json()
client_spa_sp_id = client_spa_sp["value"][0]["id"] if client_spa_sp.get("value") else None

if client_spa_sp_id and blueprint_sp_id:
    try:
        grants_url = (
            f"{GRAPH_ROOT}/oauth2PermissionGrants?"
            f"$filter=clientId eq '{client_spa_sp_id}' and resourceId eq '{blueprint_sp_id}'"
        )
        existing_consent = graph_request("GET", grants_url, admin_token).json()
        if existing_consent.get("value"):
            print("Admin consent already exists (Client SPA → Blueprint)")
        else:
            grant_body = {
                "clientId": client_spa_sp_id,
                "consentType": "AllPrincipals",
                "resourceId": blueprint_sp_id,
                "scope": "access_as_user",
            }
            graph_request(
                "POST",
                f"{GRAPH_ROOT}/oauth2PermissionGrants",
                admin_token,
                grant_body,
            )
            print("Admin consent granted for Client SPA → Blueprint")
    except RuntimeError as exc:
        print(f"Warning: {exc}")
else:
    print(
        f"Could not find service principals "
        f"(Client SPA: {client_spa_sp_id}, Blueprint: {blueprint_sp_id})"
    )

Admin consent granted for Client SPA → Blueprint
